# Pooled model comparison: one-year-ahead Ohio housing values

This notebook compares eight approaches for **ZHVI** and **ZORI** across the same nine Ohio counties. The first six models are pooled county-panel regressions with county fixed effects. ARIMA and ARIMAX are fit to each county's own level series, with one shared order selected across counties; stacking counties into one ARIMA series would create false time transitions.

| # | Model | Modeling target |
|---|---|---|
| 1 | Mean baseline | One-year growth rate, county fixed effects |
| 2 | LASSO | One-year growth rate |
| 3 | Elastic Net | One-year growth rate |
| 4 | Ridge | One-year growth rate |
| 5 | XGBoost | One-year growth rate |
| 6 | Pooled AR(1) dynamic panel | One-year growth rate; shared coefficients and county effects |
| 7 | ARIMAX | Original ZHVI/ZORI level, county-specific histories, selected exogenous inputs |
| 8 | ARIMA | Original ZHVI/ZORI level, county-specific histories |

## Common comparison rules

- **Train:** 2015Q1–2019Q4; **validation:** 2021Q1–2021Q4; **test:** 2023Q1–2024Q4.
- Before comparing models, the notebook fixes a shared set of ten predictors for models 2–6. The mean baseline is the sole predictor-free benchmark. All six pooled models include county fixed effects; seasonal versions also include Q2, Q3, and Q4 indicators with Q1 as the reference quarter.
- For models 1–6, forecast growth is converted back to the Zillow level at (t+4). Every model is then scored on the same observed (t+4) level using MAE, RMSE, and (R^2).
- For every model and each target, validation compares a version **without** and **with** quarter indicators. Hyperparameters and ARIMA specifications are selected on validation only. Test results are reported after those choices are locked.
- The ARIMAX future economic predictors are held at their last observed value for the four-quarter forecast. This is a persistence scenario because their future values are unknown at prediction time.

The level model forecasts dollars (or dollars per month for ZORI), not log levels. For numerical stability, Zillow levels are expressed in thousands of dollars inside Statsmodels and converted back to their original units after forecasting.

In [ ]:
# Imports and project paths
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from statsmodels.tsa.arima.model import ARIMA

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "processed" / "1-year-ahead split"
FEATURE_PANEL_PATH = PROJECT_ROOT / "processed" / "feature engineered data" / "master_county_quarter_features.csv"
DICTIONARY_PATH = PROJECT_ROOT / "metadata" / "master_feature_dictionary.csv"
RESULTS_DIR = PROJECT_ROOT / "results" / "checkpoint3_model_compare"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# XGBoost lives in the workspace's local package folder.
LOCAL_PYTHON_PACKAGES = PROJECT_ROOT.parent / ".python_packages"
if LOCAL_PYTHON_PACKAGES.exists():
    sys.path.insert(0, str(LOCAL_PYTHON_PACKAGES))
from xgboost import XGBRegressor

TARGETS = {
    "ZHVI": {"growth": "zhvi_growth_ahead_1y_pct", "level": "zhvi_usd"},
    "ZORI": {"growth": "zori_growth_ahead_1y_pct", "level": "zori_usd_month"},
}
COUNTY_COLUMN = "county_name"
COUNTY_KEY = "county_fips"
HORIZON_QUARTERS = 4

print(f"Project root: {PROJECT_ROOT}")
print(f"Results folder: {RESULTS_DIR}")

In [ ]:
# Load the fixed chronological split files and the complete county-quarter history.
train = pd.read_csv(SPLIT_DIR / "train.csv", dtype={"county_fips": str})
validation = pd.read_csv(SPLIT_DIR / "validation.csv", dtype={"county_fips": str})
test = pd.read_csv(SPLIT_DIR / "test.csv", dtype={"county_fips": str})
feature_panel = pd.read_csv(FEATURE_PANEL_PATH, dtype={"county_fips": str})
feature_dictionary = pd.read_csv(DICTIONARY_PATH)

for frame in [train, validation, test, feature_panel]:
    frame[COUNTY_KEY] = frame[COUNTY_KEY].astype(str).str.zfill(5)

# The project has discontinued three-year housing features and targets.
DISCONTINUED_FIELDS = {
    "zhvi_growth_prev_3y_pct", "zori_growth_prev_3y_pct",
    "zhvi_growth_ahead_3y_pct", "zori_growth_ahead_3y_pct",
}
for name, frame in [("train", train), ("validation", validation), ("test", test), ("feature panel", feature_panel)]:
    assert not (DISCONTINUED_FIELDS & set(frame.columns)), f"{name} contains discontinued 3-year fields."

# Translate each 1-year growth label into the actual Zillow level four quarters later.
# This gives all eight model families the same outcome scale for evaluation.
for frame in [train, validation, test]:
    for target_name, target_info in TARGETS.items():
        growth_col = target_info["growth"]
        current_level_col = target_info["level"]
        frame[f"{target_name.lower()}_level_t_plus_4"] = (
            frame[current_level_col] * (1 + frame[growth_col] / 100)
        )

for split_name, frame in [("Train", train), ("Validation", validation), ("Test", test)]:
    quarters = frame[["year", "quarter"]].drop_duplicates().sort_values(["year", "quarter"])
    first, last = quarters.iloc[0], quarters.iloc[-1]
    print(f"{split_name:10s}: {len(frame):3d} rows | {first.year}Q{first.quarter}–{last.year}Q{last.quarter}")
    assert frame.groupby(["year", "quarter"]).size().eq(9).all()

assert feature_panel.groupby(COUNTY_KEY).size().eq(44).all()
assert len(feature_panel) == 396

## Predictor selection policy

The common set below is fixed before the model comparison. It represents housing momentum, cross-market rent momentum, current financing and inflation conditions, local labor conditions, and affordability. It excludes future labels, calendar keys, data-quality flags, raw ACS values with unresolved publication timing, and redundant Treasury maturities. The predictor list is the same for LASSO, Elastic Net, Ridge, XGBoost, and the pooled dynamic panel; the dynamic model's autoregressive information is already represented by the lagged housing-growth features in this set.

County fixed effects and quarter indicators are categorical controls, not members of this numeric predictor set. The mean baseline uses no numeric predictors by definition; it is fit jointly with county indicators, and quarter indicators are toggled for its seasonal comparison.

For ARIMAX, we use a small set of pre-declared economic predictor bundles. Validation chooses the order and bundle together. This keeps predictor selection auditable and avoids screening features using the final test period.

In [ ]:
# Shared predictors for models 2–6; fixed before comparing validation results.
COMMON_PREDICTORS = [
    "zhvi_growth_qoq_pct",
    "zhvi_growth_prev_1y_pct",
    "zori_growth_qoq_pct",
    "zori_growth_prev_1y_pct",
    "nat_mortgage_rate_30y_pct",
    "mortgage_rate_change_prev_1y_pp",
    "nat_inflation_yoy_pct",
    "cty_unemployment_rate_pct",
    "cty_unemployment_change_prev_1y_pp",
    "price_to_rent_ratio",
]

for target_name, target_info in TARGETS.items():
    needed = COMMON_PREDICTORS + [target_info["growth"], target_info["level"]]
    missing = sorted(set(needed) - set(train.columns))
    assert not missing, f"{target_name} is missing expected columns: {missing}"

# Show descriptions and source safety notes so the chosen set can be reviewed.
feature_dictionary.loc[
    feature_dictionary["variable_name"].isin(COMMON_PREDICTORS),
    ["variable_name", "category", "source", "safe_at_prediction_time", "missingness_notes"],
].sort_values("variable_name")

## Shared pooled regression setup

Each regression is fit once across all county rows. County fixed effects are represented by county indicators with one reference county omitted. Seasonal models add Q2, Q3, and Q4 indicators with Q1 as the reference quarter. Numeric imputation and scaling are fit on the training rows only.

In [ ]:
def make_pooled_pipeline(numeric_features, estimator, use_quarter_dummies):
    """Fit one pooled county model with county fixed effects and optional quarter effects."""
    transformers = [
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), numeric_features),
        ("county_fixed_effects", OneHotEncoder(drop="first", handle_unknown="ignore"), [COUNTY_COLUMN]),
    ]
    if use_quarter_dummies:
        transformers.append((
            "quarter_fixed_effects",
            OneHotEncoder(categories=[[1, 2, 3, 4]], drop="first", handle_unknown="ignore"),
            ["quarter"],
        ))
    preprocessing = ColumnTransformer(transformers=transformers, remainder="drop")
    return Pipeline([("preprocessing", preprocessing), ("model", estimator)])


def score_levels(actual_level, predicted_level):
    return {
        "MAE": mean_absolute_error(actual_level, predicted_level),
        "RMSE": mean_squared_error(actual_level, predicted_level) ** 0.5,
        "R2": r2_score(actual_level, predicted_level),
    }


def predict_level_from_growth(growth_prediction_pct, current_level):
    """Convert a predicted one-year percentage change to a future Zillow level."""
    return np.asarray(current_level) * (1 + np.asarray(growth_prediction_pct) / 100)


def fit_predict_growth_model(training_data, prediction_data, target_name, estimator, use_quarter_dummies, numeric_features=None):
    """Fit a pooled growth model and return future level predictions."""
    target_info = TARGETS[target_name]
    growth_target = target_info["growth"]
    current_level = target_info["level"]
    numeric_features = COMMON_PREDICTORS if numeric_features is None else numeric_features
    model = make_pooled_pipeline(numeric_features, estimator, use_quarter_dummies)
    columns = numeric_features + [COUNTY_COLUMN] + (["quarter"] if use_quarter_dummies else [])
    model.fit(training_data[columns], training_data[growth_target])
    predicted_growth_pct = model.predict(prediction_data[columns])
    predicted_level = predict_level_from_growth(predicted_growth_pct, prediction_data[current_level])
    return model, predicted_level, predicted_growth_pct


def baseline_predictions(training_data, prediction_data, target_name, use_quarter_dummies):
    """County mean-growth baseline, estimated as one pooled county-FE regression."""
    model, predicted_level, predicted_growth = fit_predict_growth_model(
        training_data,
        prediction_data,
        target_name,
        LinearRegression(),
        use_quarter_dummies,
        numeric_features=[],
    )
    return model, predicted_level, predicted_growth

# Confirm that the models share the same numeric predictor list.
print(f"Shared numeric predictors for models 2–6 ({len(COMMON_PREDICTORS)}):")
print("  " + ", ".join(COMMON_PREDICTORS))
print("County fixed effects: all nine counties, one pooled fit; Q1 is the quarter reference.")

## Validation selection for models 1–6

For each target and each model, validation tests both seasonal settings. LASSO, Elastic Net, and Ridge choose their regularization parameters on validation. XGBoost uses a small shallow candidate grid. The pooled AR(1) dynamic panel is a linear pooled model using the same ten predictors, including trailing ZHVI and ZORI momentum. The mean baseline is a county-FE average-growth benchmark.

In [ ]:
# Small, regularized candidate grids for the 180-row training panel.
REGRESSION_CANDIDATES = {
    "LASSO": [Lasso(alpha=value, max_iter=20000) for value in [0.01, 0.1, 1.0, 10.0]],
    "Elastic Net": [
        ElasticNet(alpha=alpha, l1_ratio=l1_ratio, max_iter=20000)
        for alpha in [0.01, 0.1, 1.0]
        for l1_ratio in [0.25, 0.5, 0.75]
    ],
    "Ridge": [Ridge(alpha=value) for value in [0.01, 0.1, 1.0, 10.0, 100.0, 1000.0]],
    "XGBoost": [
        XGBRegressor(objective="reg:squarederror", random_state=42, n_jobs=1,
                     n_estimators=trees, max_depth=depth, learning_rate=rate,
                     min_child_weight=child, subsample=0.8, colsample_bytree=0.8,
                     reg_lambda=regularization)
        for trees, depth, rate, child, regularization in [
            (50, 1, 0.05, 10, 10),
            (100, 1, 0.05, 10, 10),
            (75, 2, 0.03, 15, 20),
        ]
    ],
}

# Parameter-free shared-slope pooled panel AR(1) dynamic regression.
REGRESSION_CANDIDATES["Pooled AR(1) dynamic panel"] = [LinearRegression()]

training_candidates = pd.concat([train, validation], ignore_index=True)
pooled_validation_records = []
pooled_validation_predictions = {}
pooled_selected = {}

for target_name, target_info in TARGETS.items():
    actual_level = validation[f"{target_name.lower()}_level_t_plus_4"].to_numpy()

    # Candidate 1: pooled county mean-growth baseline, with or without quarter effects.
    for include_quarter in [False, True]:
        _, predicted_level, _ = baseline_predictions(train, validation, target_name, include_quarter)
        result_key = (target_name, "Mean baseline", include_quarter)
        pooled_validation_predictions[result_key] = predicted_level
        pooled_validation_records.append({
            "target": target_name, "model": "Mean baseline",
            "quarter_dummies": include_quarter, "candidate": "county mean growth",
            **score_levels(actual_level, predicted_level),
        })

    # Candidates 2–6 use the same shared numeric predictors and county indicators.
    for model_name, estimators in REGRESSION_CANDIDATES.items():
        for include_quarter in [False, True]:
            candidate_rows = []
            for candidate_number, estimator in enumerate(estimators, start=1):
                _, predicted_level, _ = fit_predict_growth_model(
                    train, validation, target_name, clone(estimator), include_quarter
                )
                metrics = score_levels(actual_level, predicted_level)
                candidate_rows.append((metrics["RMSE"], metrics["MAE"], candidate_number, estimator, predicted_level, metrics))
            candidate_rows.sort(key=lambda row: (row[0], row[1]))
            _, _, chosen_number, chosen_estimator, chosen_prediction, chosen_metrics = candidate_rows[0]
            result_key = (target_name, model_name, include_quarter)
            pooled_validation_predictions[result_key] = chosen_prediction
            pooled_selected[result_key] = {"candidate_number": chosen_number, "estimator": clone(chosen_estimator)}
            pooled_validation_records.append({
                "target": target_name, "model": model_name,
                "quarter_dummies": include_quarter,
                "candidate": str(chosen_estimator), **chosen_metrics,
            })

pooled_validation_results = pd.DataFrame(pooled_validation_records)
pooled_validation_summary = (
    pooled_validation_results.sort_values(["target", "model", "RMSE"])
    .reset_index(drop=True)
)
display(pooled_validation_summary.round(3))

## ARIMA and ARIMAX order and exogenous predictor selection

ARIMA and ARIMAX model the **original Zillow level series**. For numerical stability, the level is represented in thousands of dollars during estimation; there is no logarithm or differencing of the target before fitting. Candidate orders are selected by pooled validation RMSE across all 9 counties × 4 origins, with one shared order per Zillow target. Each county's history remains a separate time series. For ARIMAX, validation also chooses one pre-declared economic predictor bundle.

ARIMAX uses a small bundle because the training history per county is short. Future exogenous values are held at their most recently observed values, while future quarter indicators are known and set to their calendar quarters. The quarter-dummy toggle is selected on validation for ARIMA and ARIMAX too.

In [ ]:
# A compact order grid limits parameter count for short county histories.
ARIMA_ORDERS = [
    (0, 0, 0), (1, 0, 0), (0, 1, 0),
    (1, 1, 0), (0, 1, 1), (1, 1, 1),
]
ARIMAX_PREDICTOR_BUNDLES = {
    "Mortgage rate": ["nat_mortgage_rate_30y_pct"],
    "Mortgage + inflation": ["nat_mortgage_rate_30y_pct", "nat_inflation_yoy_pct"],
    "Mortgage + county unemployment": ["nat_mortgage_rate_30y_pct", "cty_unemployment_rate_pct"],
}
ARIMAX_COLUMNS = sorted({column for bundle in ARIMAX_PREDICTOR_BUNDLES.values() for column in bundle})

feature_panel["period"] = pd.PeriodIndex(
    feature_panel["year"].astype(str) + "Q" + feature_panel["quarter"].astype(str), freq="Q"
)
county_histories = {}
for county_id, county_history in feature_panel.groupby(COUNTY_KEY):
    county_history = county_history.sort_values("period").set_index("period")
    county_histories[county_id] = county_history[["zhvi_usd", "zori_usd_month"] + ARIMAX_COLUMNS].copy()


def make_future_quarter_index(origin, number_of_steps):
    return pd.period_range(origin + 1, periods=number_of_steps, freq="Q")


def make_arima_exog(history, origin, order, exogenous_columns, use_quarter_dummies):
    """Create standardized observed exog and known/persisted future exog."""
    observed = history.loc[:origin]
    future_periods = make_future_quarter_index(origin, HORIZON_QUARTERS)
    parts = []

    if exogenous_columns:
        exog_observed = observed[exogenous_columns].astype(float)
        if exog_observed.isna().any().any():
            raise ValueError("Missing values in ARIMAX history")
        means = exog_observed.mean()
        scales = exog_observed.std(ddof=0).replace(0, np.nan)
        if scales.isna().any():
            raise ValueError("Constant ARIMAX predictor in history")
        standardized_observed = (exog_observed - means) / scales
        future_values = np.repeat(standardized_observed.iloc[[-1]].to_numpy(), HORIZON_QUARTERS, axis=0)
        future_economic = pd.DataFrame(future_values, index=future_periods, columns=exogenous_columns)
        parts.append((standardized_observed, future_economic))

    if use_quarter_dummies:
        quarter_observed = pd.get_dummies(observed.index.quarter, prefix="Q", dtype=float)
        quarter_observed.index = observed.index
        quarter_observed = quarter_observed.reindex(columns=["Q_2", "Q_3", "Q_4"], fill_value=0.0)
        quarter_future = pd.get_dummies(future_periods.quarter, prefix="Q", dtype=float)
        quarter_future.index = future_periods
        quarter_future = quarter_future.reindex(columns=["Q_2", "Q_3", "Q_4"], fill_value=0.0)
        parts.append((quarter_observed, quarter_future))

    if not parts:
        return None, None
    observed_frames = [pair[0] for pair in parts]
    future_frames = [pair[1] for pair in parts]
    observed_exog = pd.concat(observed_frames, axis=1).astype(float)
    future_exog = pd.concat(future_frames, axis=1).astype(float)
    return observed_exog, future_exog


def forecast_one_county_level(history, origin, target_level_column, order, exogenous_columns, use_quarter_dummies):
    """Fit a level ARIMA/ARIMAX through the origin; forecast one year ahead."""
    observed = history.loc[:origin].copy()
    level_thousands = observed[target_level_column].astype(float) / 1000.0
    observed_exog, future_exog = make_arima_exog(
        history, origin, order, exogenous_columns, use_quarter_dummies
    )
    # ARIMA with d=1 uses a linear drift; d=0 uses an intercept.
    trend = "t" if order[1] == 1 else "c"
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        fit = ARIMA(
            level_thousands,
            exog=observed_exog,
            order=order,
            trend=trend,
            enforce_stationarity=False,
            enforce_invertibility=False,
        ).fit()
        forecast = fit.get_forecast(steps=HORIZON_QUARTERS, exog=future_exog)

    predicted_mean = np.asarray(forecast.predicted_mean, dtype=float)
    interval = np.asarray(forecast.conf_int(alpha=0.05), dtype=float)
    return {
        "prediction": float(predicted_mean[-1] * 1000.0),
        "lower_95": float(interval[-1, 0] * 1000.0),
        "upper_95": float(interval[-1, 1] * 1000.0),
    }


def forecast_county_rows(rows, target_level_column, order, exogenous_columns, use_quarter_dummies):
    forecast_rows = []
    for row in rows.itertuples(index=False):
        origin = pd.Period(year=int(row.year), quarter=int(row.quarter), freq="Q")
        result = forecast_one_county_level(
            county_histories[row.county_fips], origin, target_level_column,
            order, exogenous_columns, use_quarter_dummies,
        )
        forecast_rows.append({
            "county_fips": row.county_fips, "county_name": row.county_name,
            "year": int(row.year), "quarter": int(row.quarter), **result,
        })
    return pd.DataFrame(forecast_rows)


def select_time_series_spec(target_name, family, use_quarter_dummies):
    """Choose order and, for ARIMAX, predictor bundle on validation RMSE."""
    target_level_column = TARGETS[target_name]["level"]
    actual_column = f"{target_name.lower()}_level_t_plus_4"
    bundles = {"None": []} if family == "ARIMA" else ARIMAX_PREDICTOR_BUNDLES
    candidate_rows = []

    for bundle_name, exogenous_columns in bundles.items():
        for order in ARIMA_ORDERS:
            try:
                forecasts = forecast_county_rows(
                    validation, target_level_column, order, exogenous_columns, use_quarter_dummies
                )
                actual = validation[actual_column].to_numpy()
                predicted = forecasts["prediction"].to_numpy()
                metrics = score_levels(actual, predicted)
                candidate_rows.append({
                    "target": target_name, "family": family,
                    "quarter_dummies": use_quarter_dummies,
                    "order": str(order), "p": order[0], "d": order[1], "q": order[2],
                    "predictor_bundle": bundle_name,
                    "predictors": ", ".join(exogenous_columns),
                    **metrics,
                })
            except Exception as error:
                candidate_rows.append({
                    "target": target_name, "family": family,
                    "quarter_dummies": use_quarter_dummies,
                    "order": str(order), "p": order[0], "d": order[1], "q": order[2],
                    "predictor_bundle": bundle_name,
                    "predictors": ", ".join(exogenous_columns),
                    "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                    "fit_error": f"{type(error).__name__}: {error}",
                })

    table = pd.DataFrame(candidate_rows).sort_values(["RMSE", "MAE"], na_position="last")
    valid = table.dropna(subset=["RMSE"])
    if valid.empty:
        raise RuntimeError(f"No valid {family} specification for {target_name}.")
    best = valid.iloc[0]
    bundle = [] if family == "ARIMA" else bundles[best["predictor_bundle"]]
    spec = {"order": (int(best.p), int(best.d), int(best.q)), "predictors": bundle,
            "bundle_name": best["predictor_bundle"]}
    return table, spec

# Select order (and ARIMAX exogenous bundle) separately for seasonal/nonseasonal variants.
time_series_selection_tables = []
time_series_selected = {}
for target_name in TARGETS:
    for family in ["ARIMA", "ARIMAX"]:
        for include_quarter in [False, True]:
            candidate_table, spec = select_time_series_spec(target_name, family, include_quarter)
            time_series_selection_tables.append(candidate_table)
            time_series_selected[(target_name, family, include_quarter)] = spec

time_series_selection_results = pd.concat(time_series_selection_tables, ignore_index=True)
display(
    time_series_selection_results.sort_values(["target", "family", "quarter_dummies", "RMSE"])
    .groupby(["target", "family", "quarter_dummies"], as_index=False).first()
    .round(3)
)

## Select seasonal setting and refit the eight models

The validation comparison below chooses the quarter-dummy setting separately for each target and model. That setting, plus its selected regularization/order/exogenous bundle, is then frozen for test evaluation. Test model predictions are rolling-origin forecasts: each test origin uses only Zillow and predictor observations available through that quarter.

In [ ]:
# Combine validation results for pooled models and the best ARIMA/ARIMAX candidate per seasonal setting.
selected_validation_rows = []
selected_validation_predictions = {}
selected_model_specs = {}

for target_name, target_info in TARGETS.items():
    actual_column = f"{target_name.lower()}_level_t_plus_4"
    actual_validation = validation[actual_column].to_numpy()

    # Collect the six pooled models' two seasonal variants.
    for model_name in ["Mean baseline"] + list(REGRESSION_CANDIDATES.keys()):
        for include_quarter in [False, True]:
            key = (target_name, model_name, include_quarter)
            predicted_level = pooled_validation_predictions[key]
            metrics = score_levels(actual_validation, predicted_level)
            selected_validation_predictions[key] = predicted_level
            selected_validation_rows.append({
                "target": target_name, "model": model_name,
                "quarter_dummies": include_quarter,
                "specification": ("pooled county mean" if model_name == "Mean baseline"
                                  else str(pooled_selected[key]["estimator"])),
                **metrics,
            })

    # Add best order/bundle found for each ARIMA-family seasonal setting.
    for family in ["ARIMAX", "ARIMA"]:
        for include_quarter in [False, True]:
            spec = time_series_selected[(target_name, family, include_quarter)]
            forecasts = forecast_county_rows(
                validation, target_info["level"], spec["order"], spec["predictors"], include_quarter
            )
            metrics = score_levels(actual_validation, forecasts["prediction"].to_numpy())
            model_name = family
            key = (target_name, model_name, include_quarter)
            selected_validation_predictions[key] = forecasts["prediction"].to_numpy()
            selected_model_specs[key] = spec
            selected_validation_rows.append({
                "target": target_name, "model": model_name,
                "quarter_dummies": include_quarter,
                "specification": f"order={spec['order']}; predictors={spec['bundle_name']}",
                **metrics,
            })

validation_seasonality_comparison = pd.DataFrame(selected_validation_rows)
display(
    validation_seasonality_comparison
    .sort_values(["target", "model", "quarter_dummies"])
    .round(3)
)

# Lock seasonal inclusion based only on validation RMSE for each model and target.
chosen_configuration = {}
for target_name in TARGETS:
    for model_name in ["Mean baseline"] + list(REGRESSION_CANDIDATES.keys()) + ["ARIMAX", "ARIMA"]:
        rows = validation_seasonality_comparison[
            validation_seasonality_comparison["target"].eq(target_name)
            & validation_seasonality_comparison["model"].eq(model_name)
        ].sort_values(["RMSE", "MAE"])
        best_row = rows.iloc[0]
        use_quarter = bool(best_row["quarter_dummies"])
        chosen_configuration[(target_name, model_name)] = {
            "quarter_dummies": use_quarter,
            "validation_rmse": float(best_row["RMSE"]),
        }
        if model_name in ["ARIMAX", "ARIMA"]:
            chosen_configuration[(target_name, model_name)].update(
                selected_model_specs[(target_name, model_name, use_quarter)]
            )
        elif model_name != "Mean baseline":
            chosen_configuration[(target_name, model_name)].update(
                pooled_selected[(target_name, model_name, use_quarter)]
            )

print("Validation-selected quarter-dummy setting by target and model:")
display(pd.DataFrame([
    {"target": target, "model": model, **config}
    for (target, model), config in chosen_configuration.items()
]).round(3))

In [ ]:
# Final evaluation: pooled regressions fit on train + validation; ARIMA-family forecasts roll forward by county.
train_validation = pd.concat([train, validation], ignore_index=True)
actual_columns = {target: f"{target.lower()}_level_t_plus_4" for target in TARGETS}
final_metric_rows = []
final_prediction_rows = []
time_series_interval_rows = []

for target_name, target_info in TARGETS.items():
    actual_val = validation[actual_columns[target_name]].to_numpy()
    actual_test = test[actual_columns[target_name]].to_numpy()

    for model_name in ["Mean baseline"] + list(REGRESSION_CANDIDATES.keys()):
        config = chosen_configuration[(target_name, model_name)]
        use_quarter = config["quarter_dummies"]
        if model_name == "Mean baseline":
            _, val_prediction, _ = baseline_predictions(train, validation, target_name, use_quarter)
            _, test_prediction, _ = baseline_predictions(train_validation, test, target_name, use_quarter)
            estimator_description = "pooled county mean growth"
        else:
            estimator = config["estimator"]
            _, val_prediction, _ = fit_predict_growth_model(train, validation, target_name, clone(estimator), use_quarter)
            _, test_prediction, _ = fit_predict_growth_model(train_validation, test, target_name, clone(estimator), use_quarter)
            estimator_description = str(estimator)

        for split_name, split_df, actual, prediction in [
            ("validation", validation, actual_val, val_prediction),
            ("test", test, actual_test, test_prediction),
        ]:
            final_metric_rows.append({
                "target": target_name, "model": model_name, "split": split_name,
                "quarter_dummies": use_quarter, "specification": estimator_description,
                **score_levels(actual, prediction),
            })
            final_prediction_rows.append(split_df[[COUNTY_KEY, COUNTY_COLUMN, "year", "quarter"]].assign(
                target=target_name, model=model_name, split=split_name,
                actual_level=actual, prediction_level=prediction,
            ))

    for family in ["ARIMAX", "ARIMA"]:
        config = chosen_configuration[(target_name, family)]
        use_quarter = config["quarter_dummies"]
        spec = time_series_selected[(target_name, family, use_quarter)]
        for split_name, split_df, actual in [
            ("validation", validation, actual_val), ("test", test, actual_test)
        ]:
            forecasts = forecast_county_rows(
                split_df, target_info["level"], spec["order"], spec["predictors"], use_quarter
            )
            prediction = forecasts["prediction"].to_numpy()
            final_metric_rows.append({
                "target": target_name, "model": family, "split": split_name,
                "quarter_dummies": use_quarter,
                "specification": f"order={spec['order']}; predictors={spec['bundle_name']}",
                **score_levels(actual, prediction),
            })
            final_prediction_rows.append(split_df[[COUNTY_KEY, COUNTY_COLUMN, "year", "quarter"]].assign(
                target=target_name, model=family, split=split_name,
                actual_level=actual, prediction_level=prediction,
            ))
            if split_name == "test":
                time_series_interval_rows.append(forecasts.assign(target=target_name, model=family))

metrics = pd.DataFrame(final_metric_rows)
predictions = pd.concat(final_prediction_rows, ignore_index=True)
time_series_intervals = pd.concat(time_series_interval_rows, ignore_index=True)

print("Selected model validation and test metrics (all on original Zillow levels):")
display(metrics.sort_values(["target", "split", "RMSE"]).round(3))

metrics.to_csv(RESULTS_DIR / "metrics.csv", index=False)
predictions.to_csv(RESULTS_DIR / "predictions.csv", index=False)
validation_seasonality_comparison.to_csv(RESULTS_DIR / "validation_seasonality_comparison.csv", index=False)
time_series_selection_results.to_csv(RESULTS_DIR / "arima_order_and_exogenous_selection.csv", index=False)

## Test uncertainty bands

The non-time-series models use quarter-cluster bootstrap refits: each sampled quarter keeps all nine county rows together, preserving shared national and state shocks. These bands describe coefficient/model-estimation variability for the mean county-level forecast. ARIMA and ARIMAX use their conditional four-step forecast intervals. The ARIMAX interval does not include uncertainty in future macro predictors because those predictors are held fixed at their latest observed values.

In [ ]:
# Bootstrap the selected pooled models on train + validation, resampling full quarters.
NUMBER_OF_BOOTSTRAPS = 200
rng = np.random.default_rng(2026)
training_periods = (
    train_validation[["year", "quarter"]].drop_duplicates()
    .sort_values(["year", "quarter"]).reset_index(drop=True)
)
bootstrap_prediction_rows = []

for bootstrap_number in range(NUMBER_OF_BOOTSTRAPS):
    sampled_period_indices = rng.integers(0, len(training_periods), size=len(training_periods))
    sampled_blocks = []
    for period_index in sampled_period_indices:
        period_row = training_periods.iloc[period_index]
        sampled_blocks.append(train_validation.loc[
            train_validation["year"].eq(period_row["year"])
            & train_validation["quarter"].eq(period_row["quarter"])
        ])
    bootstrap_train = pd.concat(sampled_blocks, ignore_index=True)

    for target_name in TARGETS:
        for model_name in ["Mean baseline"] + list(REGRESSION_CANDIDATES.keys()):
            config = chosen_configuration[(target_name, model_name)]
            if model_name == "Mean baseline":
                _, prediction, _ = baseline_predictions(
                    bootstrap_train, test, target_name, config["quarter_dummies"]
                )
            else:
                _, prediction, _ = fit_predict_growth_model(
                    bootstrap_train, test, target_name, clone(config["estimator"]), config["quarter_dummies"]
                )
            for row, predicted_value in zip(test.itertuples(index=False), prediction):
                bootstrap_prediction_rows.append({
                    "target": target_name, "model": model_name,
                    "county_fips": row.county_fips,
                    "year": int(row.year), "quarter": int(row.quarter),
                    "bootstrap": bootstrap_number, "prediction_level": float(predicted_value),
                })

bootstrap_predictions = pd.DataFrame(bootstrap_prediction_rows)
bootstrap_bands = (
    bootstrap_predictions.groupby(["target", "model", "year", "quarter"], as_index=False)["prediction_level"]
    .agg(lower_95=lambda values: values.quantile(0.025),
         upper_95=lambda values: values.quantile(0.975))
)

# Average county-specific conditional forecast intervals for each test quarter.
time_series_bands = (
    time_series_intervals.groupby(["target", "model", "year", "quarter"], as_index=False)
    .agg(lower_95=("lower_95", "mean"), upper_95=("upper_95", "mean"))
)
confidence_bands = pd.concat([bootstrap_bands, time_series_bands], ignore_index=True)
confidence_bands.to_csv(RESULTS_DIR / "test_confidence_bands.csv", index=False)
print(f"Built 95% bands for {confidence_bands['model'].nunique()} models across both targets.")

## Model-specific test plots

Each panel shows the average observed (t+4) Zillow level and the model's average forecast across the nine counties for each 2023Q1–2024Q4 prediction origin. The actual and predicted series are on the same original Zillow level scale. A separate figure is saved for ZHVI and ZORI.

In [ ]:
# Plot one model per panel; all panels for a target share the same level scale.
MODEL_ORDER = ["Mean baseline", "LASSO", "Elastic Net", "Ridge", "XGBoost",
               "Pooled AR(1) dynamic panel", "ARIMAX", "ARIMA"]
FIGURE_PATHS = {}

for target_name in TARGETS:
    target_predictions = predictions[
        predictions["target"].eq(target_name) & predictions["split"].eq("test")
    ].copy()
    actual_by_quarter = (
        target_predictions.drop_duplicates([COUNTY_COLUMN, "year", "quarter"])
        .groupby(["year", "quarter"], as_index=False)["actual_level"].mean()
    )
    actual_by_quarter["period"] = actual_by_quarter["year"].astype(str) + "Q" + actual_by_quarter["quarter"].astype(str)

    point_forecasts = (
        target_predictions.groupby(["model", "year", "quarter"], as_index=False)["prediction_level"].mean()
        .merge(confidence_bands[confidence_bands["target"].eq(target_name)],
               on=["model", "year", "quarter"], how="left", validate="one_to_one")
    )
    point_forecasts["period"] = point_forecasts["year"].astype(str) + "Q" + point_forecasts["quarter"].astype(str)

    number_of_columns = 3
    number_of_rows = int(np.ceil(len(MODEL_ORDER) / number_of_columns))
    figure, axes = plt.subplots(number_of_rows, number_of_columns,
                                figsize=(15, 4.1 * number_of_rows), sharex=True, sharey=True)
    axes = np.asarray(axes).reshape(-1)
    colors = dict(zip(MODEL_ORDER, plt.cm.tab10.colors[:len(MODEL_ORDER)]))

    for axis, model_name in zip(axes, MODEL_ORDER):
        forecast = point_forecasts[point_forecasts["model"].eq(model_name)]
        color = colors[model_name]
        axis.plot(actual_by_quarter["period"], actual_by_quarter["actual_level"],
                  color="black", linewidth=2.2, marker="o", label="Actual average", zorder=3)
        axis.plot(forecast["period"], forecast["prediction_level"],
                  color=color, linewidth=1.8, marker=".", label="Predicted average", zorder=2)
        axis.fill_between(forecast["period"], forecast["lower_95"], forecast["upper_95"],
                          color=color, alpha=0.16, label="95% interval", zorder=1)
        axis.set_title(model_name, fontsize=10)
        axis.grid(alpha=0.25)

    for axis in axes[len(MODEL_ORDER):]:
        axis.set_visible(False)
    axes[0].legend(fontsize=8, frameon=False, loc="best")
    for axis in axes[-number_of_columns:]:
        if axis.get_visible():
            axis.tick_params(axis="x", rotation=45)

    level_label = "Home value (USD)" if target_name == "ZHVI" else "Rent index (USD/month)"
    figure.suptitle(f"2023–2024 {target_name}: model forecasts and 95% uncertainty bands", fontsize=14, y=0.995)
    figure.supxlabel("Prediction origin")
    figure.supylabel(f"Average one-year-ahead {level_label}")
    figure.text(0.5, 0.01,
                "Pooled-model bands: quarter-cluster bootstrap; ARIMA-family bands: conditional forecast intervals.",
                ha="center", fontsize=8, color="dimgray")
    figure.tight_layout(rect=(0.02, 0.04, 1, 0.97))
    figure_path = RESULTS_DIR / f"{target_name.lower()}_test_prediction_comparison.png"
    figure.savefig(figure_path, dpi=180, bbox_inches="tight")
    FIGURE_PATHS[target_name] = figure_path
    plt.show()
    print(f"Saved: {figure_path}")

## One-year-ahead growth-rate comparison

This additional view evaluates the dataset's `*_growth_ahead_1y_pct` targets directly. Level forecasts are translated back to implied growth using each county's observed Zillow level at the prediction origin. Thus the time-series models remain level models, but all forecasts in this figure and the accompanying metrics are on the one-year growth-rate scale.

In [ ]:
# Re-express each model's level forecast as the implied one-year growth rate.
# This makes the model predictions directly comparable with the *_growth_ahead_1y_pct labels.
growth_prediction_frames = []
for target_name, target_info in TARGETS.items():
    target_key = target_name.lower()
    origin_columns = [COUNTY_KEY, "year", "quarter", target_info["level"], target_info["growth"]]
    test_origins = test[origin_columns].copy()
    target_predictions = predictions[
        predictions["target"].eq(target_name) & predictions["split"].eq("test")
    ].merge(test_origins, on=[COUNTY_KEY, "year", "quarter"], how="left", validate="many_to_one")
    target_predictions["actual_growth_pct"] = target_predictions[target_info["growth"]]
    target_predictions["predicted_growth_pct"] = (
        100 * (target_predictions["prediction_level"] / target_predictions[target_info["level"]] - 1)
    )
    growth_prediction_frames.append(target_predictions[
        [COUNTY_KEY, COUNTY_COLUMN, "year", "quarter", "target", "model",
         "actual_growth_pct", "predicted_growth_pct"]
    ])
growth_predictions = pd.concat(growth_prediction_frames, ignore_index=True)

# Report test metrics for each model on the one-year-ahead growth target.
growth_metric_rows = []
for (target_name, model_name), group in growth_predictions.groupby(["target", "model"]):
    seasonality = chosen_configuration[(target_name, model_name)]
    growth_metric_rows.append({
        "target": target_name,
        "model": model_name,
        "quarter_terms": "有季度项" if seasonality["quarter_dummies"] else "无季度项",
        "validation_RMSE_level": seasonality["validation_rmse"],
        **score_levels(group["actual_growth_pct"], group["predicted_growth_pct"]),
    })
growth_metrics = pd.DataFrame(growth_metric_rows).sort_values(["target", "RMSE"])
growth_metrics.to_csv(RESULTS_DIR / "one_year_ahead_growth_test_metrics.csv", index=False)
display(growth_metrics.round(3))

# Save the validation-selected quarter setting as a standalone lookup for easy reporting.
seasonality_selection = pd.DataFrame([
    {
        "target": target_name,
        "model": model_name,
        "quarter_terms": "有季度项" if config["quarter_dummies"] else "无季度项",
        "validation_RMSE_level": config["validation_rmse"],
    }
    for (target_name, model_name), config in chosen_configuration.items()
]).sort_values(["target", "model"])
seasonality_selection.to_csv(RESULTS_DIR / "selected_quarter_terms.csv", index=False)

# For pooled regressions, first average each bootstrap's county predictions, then take its 95% band.
# Convert every county forecast using that county's origin level before averaging.
bootstrap_growth = bootstrap_predictions.merge(
    test[[COUNTY_KEY, "year", "quarter"] + [info["level"] for info in TARGETS.values()]],
    on=[COUNTY_KEY, "year", "quarter"], how="left", validate="many_to_one",
)
bootstrap_growth_frames = []
for target_name, target_info in TARGETS.items():
    part = bootstrap_growth[bootstrap_growth["target"].eq(target_name)].copy()
    part["growth_prediction_pct"] = 100 * (
        part["prediction_level"] / part[target_info["level"]] - 1
    )
    bootstrap_growth_frames.append(part[["target", "model", "year", "quarter", "bootstrap", "growth_prediction_pct"]])
bootstrap_growth = pd.concat(bootstrap_growth_frames, ignore_index=True)
bootstrap_growth_average = (
    bootstrap_growth.groupby(["target", "model", "year", "quarter", "bootstrap"], as_index=False)
    ["growth_prediction_pct"].mean()
)
bootstrap_growth_bands = (
    bootstrap_growth_average.groupby(["target", "model", "year", "quarter"], as_index=False)
    ["growth_prediction_pct"].agg(
        lower_95=lambda values: values.quantile(0.025),
        upper_95=lambda values: values.quantile(0.975),
    )
)

# Convert county-level ARIMA/ARIMAX intervals to growth-rate intervals at each origin.
ts_growth_frames = []
for target_name, target_info in TARGETS.items():
    part = time_series_intervals[time_series_intervals["target"].eq(target_name)].merge(
        test[[COUNTY_KEY, "year", "quarter", target_info["level"]]],
        on=[COUNTY_KEY, "year", "quarter"], how="left", validate="many_to_one",
    )
    part["lower_growth_pct"] = 100 * (part["lower_95"] / part[target_info["level"]] - 1)
    part["upper_growth_pct"] = 100 * (part["upper_95"] / part[target_info["level"]] - 1)
    ts_growth_frames.append(part.assign(target=target_name)[
        ["target", "model", "year", "quarter", "lower_growth_pct", "upper_growth_pct"]
    ])
ts_growth_bands = (
    pd.concat(ts_growth_frames, ignore_index=True)
    .groupby(["target", "model", "year", "quarter"], as_index=False)
    .agg(lower_95=("lower_growth_pct", "mean"), upper_95=("upper_growth_pct", "mean"))
)
growth_bands = pd.concat([bootstrap_growth_bands, ts_growth_bands], ignore_index=True)
growth_bands.to_csv(RESULTS_DIR / "one_year_ahead_growth_test_bands.csv", index=False)

# Plot actual and predicted average growth over 2023Q1–2024Q4, one model per panel.
MODEL_ORDER = ["Mean baseline", "LASSO", "Elastic Net", "Ridge", "XGBoost",
               "Pooled AR(1) dynamic panel", "ARIMAX", "ARIMA"]
GROWTH_FIGURE_PATHS = {}
for target_name in TARGETS:
    target_frame = growth_predictions[growth_predictions["target"].eq(target_name)].copy()
    target_frame["period"] = target_frame["year"].astype(str) + "Q" + target_frame["quarter"].astype(str)
    actual_by_quarter = (
        target_frame.groupby(["year", "quarter", "period"], as_index=False)["actual_growth_pct"].mean()
    )
    average_forecasts = (
        target_frame.groupby(["model", "year", "quarter", "period"], as_index=False)
        ["predicted_growth_pct"].mean()
        .merge(growth_bands[growth_bands["target"].eq(target_name)],
               on=["model", "year", "quarter"], how="left", validate="one_to_one")
    )

    figure, axes = plt.subplots(3, 3, figsize=(15, 12), sharex=True, sharey=True)
    axes = axes.reshape(-1)
    colors = dict(zip(MODEL_ORDER, plt.cm.tab10.colors[:len(MODEL_ORDER)]))
    for axis, model_name in zip(axes, MODEL_ORDER):
        forecast = average_forecasts[average_forecasts["model"].eq(model_name)]
        color = colors[model_name]
        axis.plot(actual_by_quarter["period"], actual_by_quarter["actual_growth_pct"],
                  color="black", linewidth=2.2, marker="o", label="Actual average", zorder=3)
        axis.plot(forecast["period"], forecast["predicted_growth_pct"],
                  color=color, linewidth=1.8, marker=".", label="Predicted average", zorder=2)
        axis.fill_between(forecast["period"], forecast["lower_95"], forecast["upper_95"],
                          color=color, alpha=0.16, label="95% interval", zorder=1)
        use_quarter = chosen_configuration[(target_name, model_name)]["quarter_dummies"]
        season_label = "with quarter terms" if use_quarter else "without quarter terms"
        axis.set_title(f"{model_name}\n{season_label}", fontsize=9)
        axis.grid(alpha=0.25)
    for axis in axes[len(MODEL_ORDER):]:
        axis.set_visible(False)
    axes[0].legend(fontsize=8, frameon=False, loc="best")
    for axis in axes[-3:]:
        if axis.get_visible():
            axis.tick_params(axis="x", rotation=45)
    figure.suptitle(f"2023–2024 {target_name}: one-year-ahead growth predictions and 95% bands", fontsize=14, y=0.995)
    figure.supxlabel("Prediction origin")
    figure.supylabel(f"One-year-ahead {target_name} growth (%)")
    figure.text(0.5, 0.01,
                "Pooled-model bands: quarter-cluster bootstrap; ARIMA-family bands: conditional forecast intervals.",
                ha="center", fontsize=8, color="dimgray")
    figure.tight_layout(rect=(0.02, 0.04, 1, 0.97))
    figure_path = RESULTS_DIR / f"{target_name.lower()}_1y_ahead_growth_test_prediction_comparison.png"
    figure.savefig(figure_path, dpi=180, bbox_inches="tight")
    GROWTH_FIGURE_PATHS[target_name] = figure_path
    plt.show()
    print(f"Saved: {figure_path}")

## Reading the comparison

- **Validation seasonality table:** for every model and target, compare the RMSE rows with and without quarter indicators. The chosen setting is the lower validation RMSE, with MAE as tie-breaker.
- **Final test table:** all eight models are evaluated on the same observed future ZHVI or ZORI levels. LASSO, Elastic Net, Ridge, XGBoost, the pooled dynamic panel, and the county mean baseline predict growth first and convert it using the observed level at the prediction origin.
- The six pooled models share the predictor set; LASSO, Elastic Net, and Ridge can shrink correlated effects. County indicators ensure the panel models use one pooled fit while controlling for persistent county differences.
- ARIMA and ARIMAX use separate county histories but select one common order across all counties for each target and seasonal setting. ARIMAX's exogenous predictor and fixed-future-value assumption are reported in the selection table.
- A single 2021 validation year is a weak basis for seasonal selection. Treat chosen seasonal effects and ARIMA orders as provisional; the final 2023–2024 test result remains the main out-of-period check.